In [1]:
import polars as pl
import plotly.express as px

import plotly.io as pio
pio.renderers.default = 'iframe'  # <-- Show output in HTML

COLORS = px.colors.qualitative.G10  # <-- Discrete color scheme for graphs


In [8]:
pc = pl.read_csv('localities-au.csv', has_header=False, 
                 new_columns=['locality', 'state', 'postcode', 
                              'population', 'latitude', 'longitude']) \
.sort('state') \
.with_columns(
   population=pl.col('population').cast(pl.Int32, strict=False)
   )

In [3]:
fig = px.scatter(pc, x="longitude", y="latitude",
                 color="state", color_discrete_sequence=COLORS,
                 hover_name="locality")
fig.update_traces(marker=dict(size=2, line=dict(width=0)))  # <-- No outlines on dots
fig.show()

In [4]:
state_populations = pc.sql("""    
    SELECT state, SUM(population) as population 
    FROM self 
    GROUP BY state
""")

# Same
state_populations = pc.select(['state', 'population']) \
    .group_by('state', maintain_order=True) \
    .sum()

fig = px.bar(state_populations, x='state', y='population', 
             color="state", color_discrete_sequence=COLORS)
fig.show()